# To Run Spectral Ratio Illumination Demo on Google Colab

Omar Elmady 

Wednesday, Dec 11 

CS 7180 

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Update model link** in Step 2 if you have it from your professor
3. **Run all cells in order** (Runtime → Run all)

## What this notebook does:
- Checks GPU availability
- Downloads model from Google Drive (if link provided)
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "⚠️ No GPU detected - will use CPU (slower but works)"

## Step 2: Download Model and Clone Repository

### 2a. Download Model from Google Drive

**Ask your professor** for a Google Drive link to the model file.  
Then paste the **file ID** below.

**Example:** If link is `https://drive.google.com/file/d/1ABC123XYZ/view?usp=sharing`  
Then FILE_ID is: `1ABC123XYZ`

If you don't have the model, leave `MODEL_DRIVE_ID = None` and skip to Step 2b.

In [ ]:
import os

# ============================================
# CONFIGURATION: Paste Google Drive file ID here
# ============================================
MODEL_DRIVE_ID = "1h2fVtLQJpgLl4_C3MLA_VDuqlJTcAqf6"  # Replace with file ID from your professor's shared link
# Example: MODEL_DRIVE_ID = "1ABC123XYZ"
# ============================================

if MODEL_DRIVE_ID:
    print("📥 Downloading model from Google Drive...")
    
    # Install gdown for Drive downloads
    !pip install -q gdown
    
    import gdown
    
    # Create model directory
    os.makedirs('/content/model', exist_ok=True)
    
    # Download file
    output_path = '/content/model/UNET_run_x10_01_last_model.pth'
    url = f'https://drive.google.com/uc?id={MODEL_DRIVE_ID}'
    
    try:
        gdown.download(url, output_path, quiet=False)
        
        # Verify download
        if os.path.exists(output_path):
            size_mb = os.path.getsize(output_path) / (1024 * 1024)
            if size_mb > 100:  # Should be ~528MB
                print(f"\n✓ Model downloaded successfully: {size_mb:.1f} MB")
            else:
                print(f"\n⚠️ Model file seems too small ({size_mb:.1f} MB)")
                print("   Check if the Drive link allows public access")
        else:
            print("\n❌ Model download failed")
            print("   Make sure the file is shared with 'Anyone with the link'")
    except Exception as e:
        print(f"\n❌ Error downloading model: {e}")
        print("   Double-check the file ID and sharing permissions")
else:
    print("⚠️ No model file ID provided")
    print("   Will run baseline-only experiments (no neural ISD prediction)")
    print("\nTo use the model:")
    print("   1. Get Google Drive link from your professor")
    print("   2. Extract file ID from the link")
    print("   3. Update MODEL_DRIVE_ID above")
    print("   4. Re-run this cell")

### 2b. Clone Repository

In [ ]:
# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
print("📥 Cloning repository from GitHub...")
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

# Copy model from /content/model if it was downloaded in Step 2a
if os.path.exists('/content/model/UNET_run_x10_01_last_model.pth'):
    import shutil
    os.makedirs('model', exist_ok=True)
    shutil.copy('/content/model/UNET_run_x10_01_last_model.pth', 
                'model/UNET_run_x10_01_last_model.pth')
    size_mb = os.path.getsize('model/UNET_run_x10_01_last_model.pth') / (1024 * 1024)
    print(f"✓ Model copied to project: {size_mb:.1f} MB")
else:
    print("⚠️ No model file available - will run baseline-only experiments")

# Show directory structure
print("\n📁 Repository contents:")
!ls -lh

## Step 3: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (full version with GUI support)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess
import sys

print("📦 Installing dependencies...\n")

# Upgrade pip
print("1️⃣ Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv (full version, not headless - Colab supports GUI)
print("\n2️⃣ Installing OpenCV, NumPy, Matplotlib...")
!pip install --quiet opencv-python numpy matplotlib

# Install PyTorch with GPU support if available
print("\n3️⃣ Installing PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\n4️⃣ Installing remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\n✓ All dependencies installed successfully!")

# Verify installations
print("\n📋 Checking installed versions:")
import torch
import cv2
import numpy as np
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")

## Step 4: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 5: Run Experiments

This cell processes all images in `data/images/` with your algorithms.

The model was downloaded automatically in Step 2, so this will run the full pipeline:
- Neural ISD prediction
- SR-constrained Retinex
- Baseline Retinex (for comparison)
- SR-based color correction

In [ ]:
import os

%cd /content/Spectral_Ratio_Illumination_Demo

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("🚀 Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --iterations 5 \
        --sigma 15 \
        --distance 1.0
else:
    print("🚀 Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   ⚠️ Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --iterations 5 \
        --sigma 15

print("\n✓ Processing complete! Check results/ directory.")

## Step 6: View Sample Results (Optional)

Display a few output images to verify processing worked correctly.

In [ ]:
import os
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Find PNG outputs in results directory
png_files = glob.glob('/content/Spectral_Ratio_Illumination_Demo/results/*.png')

if png_files:
    print(f"📸 Found {len(png_files)} output images. Showing first 3:\n")
    for img_path in png_files[:3]:
        print(f"   {os.path.basename(img_path)}")
        display(Image(filename=img_path, width=600))
        print()
else:
    print("⚠️ No PNG outputs found in results/")
    print("   Check if processing completed successfully above.")

## Step 7: Package and Download Results

This creates a `.tar.gz` archive of all results and downloads it to your local machine.

In [ ]:
import os
from google.colab import files

%cd /content/Spectral_Ratio_Illumination_Demo

# Determine what to package
has_results = os.path.exists('results')
has_tuning = os.path.exists('results_tuning')

if has_tuning:
    # If tuning was run, package all tuning results
    print("📦 Packaging parameter tuning results...")
    !tar -czf results_tuning.tar.gz results_tuning/ 2>/dev/null
    
    if os.path.exists('results_tuning.tar.gz'):
        size_mb = os.path.getsize('results_tuning.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results_tuning.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results_tuning.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results_tuning.tar.gz")
    else:
        print("⚠️ Failed to create tuning archive")

elif has_results:
    # If only single run results exist, package those
    print("📦 Packaging results...")
    !tar -czf results.tar.gz results/ 2>/dev/null
    
    if os.path.exists('results.tar.gz'):
        size_mb = os.path.getsize('results.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results.tar.gz")
    else:
        print("⚠️ Failed to create results archive")

else:
    print("⚠️ No results to package. Make sure Step 5 or parameter tuning completed successfully.")

---

## Optional: Automated Parameter Tuning

Run systematic experiments with different parameter combinations to find optimal settings.

**What this does:**
- Tests multiple values for iterations, sigma (blur), and distance (color correction)
- Saves results for each combination separately
- Helps you find the best parameters for your images

**Note**: This will take longer (proportional to number of parameter combinations × number of images)

In [ ]:
import os
import shutil
from datetime import datetime

%cd /content/Spectral_Ratio_Illumination_Demo

# Define parameter ranges to test
iterations_values = [3, 5, 10]  # Number of Retinex iterations
sigma_values = [10, 15, 20, 25]  # Gaussian blur sigma (larger = more smoothing)
distance_values = [0.5, 1.0, 1.5]  # SR color correction distance

print("🔬 Starting Automated Parameter Tuning")
print(f"   Testing {len(iterations_values)} × {len(sigma_values)} × {len(distance_values)} = {len(iterations_values) * len(sigma_values) * len(distance_values)} combinations\n")

# Create a directory for tuning results
tuning_dir = 'results_tuning'
os.makedirs(tuning_dir, exist_ok=True)

total_runs = len(iterations_values) * len(sigma_values) * len(distance_values)
current_run = 0

# Test each combination
for iterations in iterations_values:
    for sigma in sigma_values:
        for distance in distance_values:
            current_run += 1
            print(f"\n{'='*70}")
            print(f"Run {current_run}/{total_runs}: iterations={iterations}, sigma={sigma}, distance={distance}")
            print('='*70)
            
            # Run the batch processor with these parameters
            !python scripts/run_batch.py \
                --use-model \
                --retinex \
                --baseline-retinex \
                --sr-correct \
                --iterations {iterations} \
                --sigma {sigma} \
                --distance {distance}
            
            # Move results to a labeled subdirectory
            result_subdir = f"{tuning_dir}/iter{iterations}_sigma{sigma}_dist{distance}"
            if os.path.exists('results'):
                if os.path.exists(result_subdir):
                    shutil.rmtree(result_subdir)
                shutil.copytree('results', result_subdir)
                print(f"✓ Results saved to: {result_subdir}")
            
            # Clean up results directory for next run
            if os.path.exists('results'):
                shutil.rmtree('results')

print(f"\n\n{'='*70}")
print("✓ Parameter tuning complete!")
print(f"   All results saved in: {tuning_dir}/")
print('='*70)

# Show summary of all runs
print("\n📊 Summary of parameter combinations tested:")
for iterations in iterations_values:
    for sigma in sigma_values:
        for distance in distance_values:
            result_subdir = f"{tuning_dir}/iter{iterations}_sigma{sigma}_dist{distance}"
            if os.path.exists(result_subdir):
                num_files = len([f for f in os.listdir(result_subdir) if f.endswith('.png')])
                print(f"   ✓ iter={iterations:2d}, sigma={sigma:2d}, dist={distance:.1f} → {num_files} output images")

print("\n💡 Next steps:")
print("   1. Review images in each subdirectory")
print("   2. Compare visual quality across parameter settings")
print("   3. Select the best combination for your final results")
print("   4. Run Step 7 below to download all tuning results")

## Optional: Visual Comparison of Parameter Tuning Results

Display side-by-side comparisons of different parameter settings for the same image.

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

tuning_dir = 'results_tuning'

if not os.path.exists(tuning_dir):
    print("⚠️ No tuning results found. Run the parameter tuning cell above first.")
else:
    # Find all subdirectories (one per parameter combination)
    subdirs = sorted([d for d in os.listdir(tuning_dir) if os.path.isdir(os.path.join(tuning_dir, d))])
    
    if not subdirs:
        print("⚠️ No result subdirectories found in results_tuning/")
    else:
        print(f"📊 Found {len(subdirs)} parameter combinations\n")
        
        # Pick one image to compare (use first image found)
        sample_subdir = os.path.join(tuning_dir, subdirs[0])
        sample_images = glob.glob(os.path.join(sample_subdir, '*_retinex.png'))
        
        if not sample_images:
            print("⚠️ No output images found in results")
        else:
            # Get the base name of the first image (without path and suffix)
            sample_basename = os.path.basename(sample_images[0])
            print(f"Comparing results for: {sample_basename}\n")
            
            # Find this image across all parameter combinations
            comparison_images = []
            labels = []
            
            for subdir in subdirs:
                img_path = os.path.join(tuning_dir, subdir, sample_basename)
                if os.path.exists(img_path):
                    comparison_images.append(img_path)
                    labels.append(subdir.replace('iter', 'i=').replace('_sigma', ' σ=').replace('_dist', ' d='))
            
            # Display in a grid
            n_images = len(comparison_images)
            if n_images > 0:
                cols = min(3, n_images)
                rows = (n_images + cols - 1) // cols
                
                fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 6*rows))
                if n_images == 1:
                    axes = [axes]
                else:
                    axes = axes.flatten() if rows > 1 else axes
                
                for idx, (img_path, label) in enumerate(zip(comparison_images, labels)):
                    img = Image.open(img_path)
                    axes[idx].imshow(img)
                    axes[idx].set_title(label, fontsize=10)
                    axes[idx].axis('off')
                
                # Hide unused subplots
                for idx in range(n_images, rows * cols):
                    axes[idx].axis('off')
                
                plt.tight_layout()
                plt.show()
                
                print(f"\n✓ Showing {n_images} parameter variations")
                print("💡 Look for:")
                print("   - Best detail preservation (higher iterations)")
                print("   - Smoothest illumination (higher sigma)")
                print("   - Natural color appearance (adjust distance)")
            else:
                print("⚠️ No matching images found across parameter combinations")

---

## Troubleshooting

### Model download failed from Google Drive
**Causes:**
- File ID is incorrect
- File is not shared publicly ("Anyone with the link")
- Download quota exceeded

**Solutions:**
1. Verify the Drive link your professor gave you works in a browser
2. Extract the correct file ID (the long string between `/d/` and `/view`)
3. Make sure file permissions are set to "Anyone with the link can view"
4. If quota exceeded, wait a few hours or ask professor for a different link

### I don't have the model file
**Option 1:** Email your professor:
```
Subject: Request for Model Checkpoint File

Hi Professor,

I need the model checkpoint file (UNET_run_x10_01_last_model.pth, ~528MB) 
to run experiments for the CV project.

Could you please share it via Google Drive and set permissions to 
"Anyone with the link can view"?

Thank you!
```

**Option 2:** Run baseline experiments without the model
- Change Step 5 to only run `--baseline-retinex`
- You can still compare standard vs SR-constrained Retinex
- Just won't have neural ISD prediction

### Out of memory error
- Switch to CPU runtime (Runtime → Change runtime type → None)
- Or reduce batch size if processing many images

### Import errors
- Re-run Step 3 (dependency installation)
- Make sure PyTorch and OpenCV installed successfully
- Check for error messages in the output

### No results generated
- Check that `data/images/` contains .tif or .tiff files
- Verify Step 5 ran without errors
- Look for error messages in the cell output

### Colab disconnects during long runs
- Colab free tier has runtime limits (~12 hours)
- Keep the browser tab open
- Use "Runtime → Manage sessions" to monitor

### Need help?
- See `TUNING_GUIDE.md` and `README_EXPERIMENTS.md` in the repository
- Check `preflight_check.py` output for specific issues
- All documentation is in the cloned repo at `/content/Spectral_Ratio_Illumination_Demo/`